# Шаг 2. YOLO смотрит в веб-камеру в реальном времени

В первом шаге модель разбирала одну фотографию. Видео — это просто **много фотографий (кадров) подряд**.
Если успевать обрабатывать кадры достаточно быстро, получится распознавание «вживую».

Что мы сделаем:
1. Загрузим ту же модель, что и в шаге 1.
2. Проверим, что камера работает: снимем один кадр и посмотрим на него прямо в ноутбуке.
3. Запустим цикл: взять кадр → распознать → нарисовать рамки → показать в окне → повторить.
4. Будем измерять скорость (FPS) и выводить список найденных объектов на русском.

> Ноутбук нужно запускать **на своём компьютере** (VS Code или Jupyter), где есть веб-камера.
> В облачных сервисах вроде Google Colab окно `cv2.imshow` не откроется: там откройте `02_webcam_colab.ipynb`.

## 1. Импорт библиотек

- `cv2` (библиотека **OpenCV**) — работа с камерой, окнами и рисованием на картинках.
- `time` — замер времени, чтобы посчитать скорость.
- `Counter` — удобный «счётчик»: посчитает, сколько людей, сколько чашек и т.д.
- `YOLO` и `ru` — как в шаге 1.

In [ ]:
import time
from collections import Counter

import cv2
from ultralytics import YOLO

from labels_ru import ru

## 2. Модель и номер камеры

Камеры в компьютере нумеруются с нуля: `0` — обычно встроенная камера ноутбука,
`1`, `2`… — подключённые дополнительно (USB-камера, виртуальная камера и т.п.).

In [ ]:
MODEL = "yolov8n.pt"
CAMERA = 0  # если камер несколько, попробуйте 1

model = YOLO(MODEL)

## 3. Проверяем камеру: один кадр

Прежде чем запускать бесконечный цикл, убедимся, что камера вообще открывается.

- `cv2.VideoCapture(CAMERA)` — подключается к камере.
- `cap.read()` — берёт один кадр. Возвращает два значения: `ok` (получилось ли) и сам кадр `frame`.
- `cap.release()` — **обязательно** отпускаем камеру, иначе другие программы (и следующая ячейка) не смогут её открыть.

Кадр распознаём так же, как фото в шаге 1, и показываем прямо в ноутбуке.

In [ ]:
from PIL import Image

cap = cv2.VideoCapture(CAMERA)
ok, frame = cap.read()
cap.release()

if not ok:
    raise RuntimeError("Камера не открылась: проверьте подключение или номер CAMERA")

result = model(frame, conf=0.40, verbose=False)[0]
Image.fromarray(result.plot()[..., ::-1])  # BGR -> RGB, как в шаге 1

## 4. Распознавание в реальном времени

Теперь главное — цикл, который крутится, пока мы не нажмём `q`. На каждом проходе:

1. **Берём кадр** с камеры (`cap.read()`).
2. **Распознаём** его моделью. `verbose=False` отключает подробный вывод, иначе ноутбук завалит текстом.
3. **Рисуем рамки** — `result.plot()`.
4. **Считаем FPS** (frames per second, кадров в секунду): засекаем, сколько секунд прошло с прошлого кадра,
   и делим 1 на это время. Например, 0.05 с на кадр → 20 FPS. Пишем FPS и текущий порог прямо на картинке (`cv2.putText`).
5. **Пишем, что в кадре**, по-русски, например `человек: 2, чашка: 1`. Чтобы не печатать одно и то же
   30 раз в секунду, выводим строку только когда **состав меняется**.
6. **Показываем кадр** в отдельном окне (`cv2.imshow`).
7. **Проверяем клавиши** (`cv2.waitKey(1)` ждёт нажатия 1 миллисекунду):

| Клавиша | Действие |
|---------|----------|
| `+` / `-` | поднять / опустить порог уверенности на 0.05 |
| `s` | сохранить кадр в файл `shot_01.jpg`, `shot_02.jpg`… (пригодится для таблицы ошибок) |
| `q` | выход |

> Клавиши работают, только когда **активно окно с видео** (щёлкните по нему мышкой), а не ноутбук.

Весь цикл обёрнут в функцию с `try / finally`: что бы ни случилось (выход по `q`, ошибка или кнопка «Стоп» в ноутбуке),
камера будет отпущена, а окно закрыто.

In [ ]:
def run_webcam(conf=0.40):
    cap = cv2.VideoCapture(CAMERA)
    if not cap.isOpened():
        raise RuntimeError("Камера не открылась: проверьте подключение или номер CAMERA")

    shot = 0
    last_report = ""
    prev = time.time()

    try:
        while True:
            # 1. Берём кадр
            ok, frame = cap.read()
            if not ok:
                break

            # 2–3. Распознаём и рисуем рамки с подписями
            result = model(frame, conf=conf, verbose=False)[0]
            picture = result.plot()

            # 4. Скорость: сколько кадров в секунду успевает обработать компьютер
            now = time.time()
            fps = 1 / (now - prev)
            prev = now
            cv2.putText(picture, f"FPS {fps:4.1f}   conf {conf:.2f}", (10, 30),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)

            # 5. Что в кадре — по-русски, только когда состав меняется
            counts = Counter(result.names[int(c)] for c in result.boxes.cls)
            report = ", ".join(f"{ru(n)}: {k}" for n, k in counts.most_common())
            if report != last_report:
                print(report or "никого")
                last_report = report

            # 6. Показываем кадр в окне
            cv2.imshow("YOLO", picture)

            # 7. Клавиши
            key = cv2.waitKey(1) & 0xFF
            if key == ord("q"):
                break
            if key in (ord("+"), ord("=")):  # "=" — та же клавиша, что "+", но без Shift
                conf = min(conf + 0.05, 0.95)
            if key == ord("-"):
                conf = max(conf - 0.05, 0.05)
            if key == ord("s"):
                shot += 1
                cv2.imwrite(f"shot_{shot:02d}.jpg", picture)
                print(f"Сохранено: shot_{shot:02d}.jpg")
    finally:
        cap.release()
        cv2.destroyAllWindows()
        cv2.waitKey(1)  # на Windows окно иногда не закрывается без этого

Запускаем! Откроется окно **YOLO** с изображением с камеры. Чтобы выйти, щёлкните по окну и нажмите `q`.

In [ ]:
run_webcam(conf=0.40)

## Попробуйте сами

1. Посмотрите на FPS. Сколько кадров в секунду успевает ваш компьютер? Замените модель на `yolov8s.pt` — как изменится скорость?
2. Опускайте порог клавишей `-`, пока не появятся ошибки. При каком пороге модель начинает «видеть» то, чего нет?
3. Покажите камере предметы под разными углами, частично закройте их рукой. Когда модель перестаёт их узнавать?
   Сохраняйте такие кадры клавишей `s` — это материал для таблицы ошибок.
4. Измените функцию так, чтобы она печатала сообщение, когда в кадре появляется **телефон** (`"cell phone"`).